In [ ]:
# Fine-tuning Pre-trained RuBisCO Diffusion Model
**Loading existing model and fine-tuning on thermophilic dataset**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import ast
import datetime

# Device Setup

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

Using device: cpu


# Load Pre-trained Model
**Loading the existing rubisco_diffusion_model.pth**

In [ ]:
# Load the pre-trained model
model_path = "best_enhanced_rubisco_model.pth"
print(f"Loading pre-trained model from: {model_path}")

# Load the saved model state
checkpoint = torch.load(model_path, map_location=device)
print("✓ Model checkpoint loaded successfully")

# Extract model configuration
model_config = checkpoint['model_config']
vocab_size = model_config['vocab_size']
d_model = model_config['d_model']
num_heads = model_config['num_heads']
num_layers = model_config['num_layers']
d_ff = model_config['d_ff']
dropout = model_config['dropout']
max_len = model_config['max_len']
T = model_config['T']

# Extract vocabularies
aa_to_idx = checkpoint['aa_to_idx']
idx_to_aa = checkpoint['idx_to_aa']
ss_to_idx = checkpoint.get('ss_to_idx', {'H': 0, 'E': 1, 'C': 2, 'G': 3, 'I': 4, 'B': 5, 'T': 6, 'S': 7, '-': 8})

print(f"\nModel Configuration:")
print(f"  Vocabulary size: {vocab_size}")
print(f"  Model dimension: {d_model}")
print(f"  Max sequence length: {max_len}")
print(f"  Diffusion steps: {T}")
print(f"  AA vocabulary: {len(aa_to_idx)} amino acids")
print(f"  SS vocabulary: {len(ss_to_idx)} secondary structure types")

# Reconstruct Model Architecture
**Rebuilding the same architecture as the pre-trained model**

In [ ]:
# Reconstruct the exact same model architecture
print("Reconstructing model architecture...")

# Embedding layers
embedding = nn.Embedding(vocab_size, d_model, padding_idx=0).to(device)
pos_encoding = torch.nn.Parameter(torch.zeros(1, max_len, d_model, device=device), requires_grad=True)

# Feature projection layers
coordinate_projection = nn.Linear(3, d_model // 6).to(device)
angle_projection = nn.Linear(2, d_model // 6).to(device)
neighbor_projection = nn.Linear(1, d_model // 6).to(device)
confidence_projection = nn.Linear(1, d_model // 6).to(device)
ss_embedding = nn.Embedding(len(ss_to_idx), d_model // 6).to(device)
tm_projection = nn.Linear(1, d_model // 6).to(device)
feature_projection = nn.Linear(d_model + 6 * (d_model // 6), d_model).to(device)

# Time embedding
time_mlp = nn.Sequential(
    nn.Linear(d_model, d_model * 4),
    nn.SiLU(),
    nn.Linear(d_model * 4, d_model)
).to(device)

# Transformer
encoder_layer = nn.TransformerEncoderLayer(d_model=d_model,
                                           nhead=num_heads,
                                           dim_feedforward=d_ff,
                                           dropout=dropout,
                                           batch_first=True).to(device)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers).to(device)

# Output heads
output_layer = nn.Linear(d_model, vocab_size).to(device)
coordinate_head = nn.Sequential(
    nn.Linear(d_model, d_model // 2),
    nn.ReLU(),
    nn.Linear(d_model // 2, 3)
).to(device)
secondary_structure_head = nn.Sequential(
    nn.Linear(d_model, d_model // 2),
    nn.ReLU(),
    nn.Linear(d_model // 2, len(ss_to_idx))
).to(device)
angles_head = nn.Sequential(
    nn.Linear(d_model, d_model // 2),
    nn.ReLU(),
    nn.Linear(d_model // 2, 2)
).to(device)

print("✓ Model architecture reconstructed")

In [ ]:
# Load pre-trained weights
print("Loading pre-trained weights...")

embedding.load_state_dict(checkpoint['embedding_state_dict'])
time_mlp.load_state_dict(checkpoint['time_mlp_state_dict'])
transformer_encoder.load_state_dict(checkpoint['transformer_encoder_state_dict'])
output_layer.load_state_dict(checkpoint['output_layer_state_dict'])

# Load structural projection layers if they exist
if 'coordinate_projection_state_dict' in checkpoint:
    coordinate_projection.load_state_dict(checkpoint['coordinate_projection_state_dict'])
    angle_projection.load_state_dict(checkpoint['angle_projection_state_dict'])
    neighbor_projection.load_state_dict(checkpoint['neighbor_projection_state_dict'])
    confidence_projection.load_state_dict(checkpoint['confidence_projection_state_dict'])
    feature_projection.load_state_dict(checkpoint['feature_projection_state_dict'])
    print("✓ Structural feature projections loaded")

# Load additional heads if they exist
if 'coordinate_head_state_dict' in checkpoint:
    coordinate_head.load_state_dict(checkpoint['coordinate_head_state_dict'])
    secondary_structure_head.load_state_dict(checkpoint['secondary_structure_head_state_dict'])
    angles_head.load_state_dict(checkpoint['angles_head_state_dict'])
    print("✓ Multi-task output heads loaded")

if 'ss_embedding_state_dict' in checkpoint:
    ss_embedding.load_state_dict(checkpoint['ss_embedding_state_dict'])
    print("✓ Secondary structure embedding loaded")

if 'tm_projection_state_dict' in checkpoint:
    tm_projection.load_state_dict(checkpoint['tm_projection_state_dict'])
    print("✓ TM projection loaded")

# Load positional encoding
pos_encoding.data = checkpoint['pos_encoding'].data

print("✅ All pre-trained weights loaded successfully!")
print("Model is ready for fine-tuning")

# Load Thermophilic Dataset
**Loading the new dataset for fine-tuning**

In [ ]:
# Load thermophilic dataset
thermo_data_path = "thermo_data1.csv"  # Change this to your actual file path
print(f"Loading thermophilic dataset from: {thermo_data_path}")

thermo_df = pd.read_csv(thermo_data_path)
print(f"✓ Loaded {len(thermo_df)} thermophilic proteins")
print(f"Columns: {thermo_df.columns.tolist()}")
print("\nDataset info:")
thermo_df.info()
print("\nFirst few rows:")
thermo_df.head()

# Data Preprocessing for Fine-tuning
**Preparing thermophilic data in the same format as original training**

In [ ]:
# SIMPLIFIED preprocessing - just check TM values
print("Checking data...")

# Check TM values and fix if needed
if 'tm_c' in thermo_df.columns:
    # Convert TM values to numeric, handle any string issues
    thermo_df['tm_c'] = pd.to_numeric(thermo_df['tm_c'], errors='coerce')
    thermo_df['tm_c'] = thermo_df['tm_c'].fillna(70.0)  # Fill NaN with default
    
    print(f"✓ TM temperature range: {thermo_df['tm_c'].min():.1f}°C to {thermo_df['tm_c'].max():.1f}°C")
    print(f"✓ Average TM: {thermo_df['tm_c'].mean():.1f}°C")
else:
    print("⚠️ No 'tm_c' column found - will use default TM values")
    print(f"Available columns: {thermo_df.columns.tolist()}")

print(f"\nDataset summary:")
print(f"  Sequences: {len(thermo_df)}")
print(f"  Average sequence length: {thermo_df['sequence'].str.len().mean():.1f}")
print(f"  Sequence length range: {thermo_df['sequence'].str.len().min()} to {thermo_df['sequence'].str.len().max()}")
print("✓ Using simplified approach with dummy structural features for fine-tuning")

In [ ]:
# Simple data preparation function - FIXED!
def prepare_structural_data(dataframe, aa_to_idx, max_len=None):
    """Prepare data for fine-tuning - SIMPLIFIED AND ROBUST"""
    sequences = []
    tm_values = []
    
    if max_len is None:
        max_len = max(len(row['sequence']) for _, row in dataframe.iterrows())
        max_len = ((max_len + 4) // 5) * 5
    
    for idx, row in dataframe.iterrows():
        seq = row['sequence']
        
        # FIX: Convert TM to float properly
        try:
            if 'tm_c' in row:
                tm_val = float(row['tm_c'])
            else:
                tm_val = 70.0  # Default
        except (ValueError, TypeError):
            tm_val = 70.0  # Default if conversion fails
        
        # Encode sequence
        encoded_seq = [aa_to_idx.get(aa, 0) for aa in seq]
        
        # Truncate or pad to max_len
        if len(encoded_seq) > max_len:
            encoded_seq = encoded_seq[:max_len]
        else:
            pad_len = max_len - len(encoded_seq)
            encoded_seq.extend([0] * pad_len)
        
        sequences.append(encoded_seq)
        tm_values.append(tm_val)  # Just single value per sequence
    
    # Create simple dummy structural features
    batch_size = len(sequences)
    dummy_coords = [[(0.0, 0.0, 0.0)] * max_len for _ in range(batch_size)]
    dummy_angles = [[0.0] * max_len for _ in range(batch_size)]
    dummy_ss = [[2] * max_len for _ in range(batch_size)]  # All coil
    dummy_confidence = [[1.0] * max_len for _ in range(batch_size)]
    dummy_neighbors = [[0] * max_len for _ in range(batch_size)]
    
    return {
        'sequences': torch.tensor(sequences, dtype=torch.long, device=device),
        'coordinates': torch.tensor(dummy_coords, dtype=torch.float, device=device),
        'phi_angles': torch.tensor(dummy_angles, dtype=torch.float, device=device),
        'psi_angles': torch.tensor(dummy_angles, dtype=torch.float, device=device),
        'neighbor_counts': torch.tensor(dummy_neighbors, dtype=torch.float, device=device),
        'confidence_masks': torch.tensor(dummy_confidence, dtype=torch.float, device=device),
        'secondary_structures': torch.tensor(dummy_ss, dtype=torch.long, device=device),
        'tm_values': torch.tensor([[tm] * max_len for tm in tm_values], dtype=torch.float, device=device)
    }, max_len

# Split data for fine-tuning
train_thermo_df, val_thermo_df = train_test_split(thermo_df, test_size=0.2, random_state=42)
print(f"Fine-tuning split: {len(train_thermo_df)} train, {len(val_thermo_df)} validation")

# Prepare data tensors
print("Preparing fine-tuning data...")
finetune_train_data, finetune_max_len = prepare_structural_data(train_thermo_df, aa_to_idx, max_len)
finetune_val_data, _ = prepare_structural_data(val_thermo_df, aa_to_idx, finetune_max_len)

print(f"✓ Fine-tuning data prepared")
print(f"  Max length used: {finetune_max_len}")
print(f"  Original model max length: {max_len}")
print(f"  Training samples: {len(finetune_train_data['sequences'])}")
print(f"  Validation samples: {len(finetune_val_data['sequences'])}")

# Recreate Model Functions
**Recreating necessary functions for fine-tuning**

In [ ]:
# Recreate diffusion schedule
betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1 - alphas_cumprod)

# Essential functions for training
def create_time_embedding(timestep, dim):
    half_dim = dim // 2
    emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
    emb = torch.exp(torch.arange(half_dim, device=timestep.device) * -emb)
    emb = timestep[:, None] * emb[None, :]
    emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
    return emb

def sample_timesteps(batch_size, T):
    return torch.randint(1, T + 1, (batch_size,), device=device)

def add_noise(x, t, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod, vocab_size):
    batch_size, seq_len = x.shape
    alpha_t = sqrt_alphas_cumprod[t - 1]
    noise_mask = torch.bernoulli(1 - alpha_t.unsqueeze(1).expand(-1, seq_len))
    noise_tokens = torch.randint(1, vocab_size, (batch_size, seq_len), device=x.device)
    x_noisy = x * (1 - noise_mask.long()) + noise_tokens * noise_mask.long()
    return x_noisy, noise_mask

def enhanced_rubisco_diffusion_forward(x, t, mask, coords=None, angles=None, neighbors=None, confidence=None, secondary_structure=None, tm_values=None):
    """Enhanced forward pass for fine-tuning"""
    time_emb = create_time_embedding(t.float(), d_model)
    time_emb = time_mlp(time_emb)
    
    x_emb = embedding(x)
    x_emb = x_emb + pos_encoding[:, :x.size(1), :]
    
    feature_embeddings = []
    
    if coords is not None:
        coord_emb = coordinate_projection(coords)
        feature_embeddings.append(coord_emb)
    
    if angles is not None:
        angle_emb = angle_projection(angles)
        feature_embeddings.append(angle_emb)
    
    if neighbors is not None:
        neighbor_emb = neighbor_projection(neighbors)
        feature_embeddings.append(neighbor_emb)
    
    if confidence is not None:
        conf_emb = confidence_projection(confidence)
        feature_embeddings.append(conf_emb)
    
    if secondary_structure is not None:
        ss_emb = ss_embedding(secondary_structure)
        feature_embeddings.append(ss_emb)
    
    if tm_values is not None:
        tm_emb = tm_projection(tm_values.unsqueeze(-1))
        feature_embeddings.append(tm_emb)
    
    if feature_embeddings:
        all_features = torch.cat([x_emb] + feature_embeddings, dim=-1)
        x_emb = feature_projection(all_features)
    
    time_emb = time_emb.unsqueeze(1)
    x_emb = x_emb + time_emb
    
    src_key_padding_mask = (mask == 0)
    transformer_out = transformer_encoder(x_emb, src_key_padding_mask=src_key_padding_mask)
    
    sequence_logits = output_layer(transformer_out)
    coord_pred = coordinate_head(transformer_out)
    ss_pred = secondary_structure_head(transformer_out)
    angles_pred = angles_head(transformer_out)
    
    return {
        'sequence_logits': sequence_logits,
        'coord_pred': coord_pred, 
        'ss_pred': ss_pred,
        'angles_pred': angles_pred
    }

print("✓ Model functions recreated for fine-tuning")

# Fine-tuning Training
**Training the pre-trained model on thermophilic data**

In [ ]:
# Setup optimizer for fine-tuning with lower learning rate
finetune_optimizer = optim.AdamW(
    list(embedding.parameters()) + 
    list(time_mlp.parameters()) + 
    list(transformer_encoder.parameters()) + 
    list(output_layer.parameters()) + 
    list(coordinate_projection.parameters()) +
    list(angle_projection.parameters()) +
    list(neighbor_projection.parameters()) +
    list(confidence_projection.parameters()) +
    list(ss_embedding.parameters()) +
    list(tm_projection.parameters()) +
    list(feature_projection.parameters()) +
    list(coordinate_head.parameters()) +
    list(secondary_structure_head.parameters()) +
    list(angles_head.parameters()) +
    [pos_encoding], 
    lr=5e-5,  # Lower learning rate for fine-tuning
    weight_decay=1e-5
)

# Loss functions
sequence_loss_fn = nn.CrossEntropyLoss(ignore_index=0)
coord_loss_fn = nn.MSELoss()
ss_loss_fn = nn.CrossEntropyLoss(ignore_index=8)
angles_loss_fn = nn.MSELoss()

def compute_multi_task_loss(predictions, targets, mask):
    """Multi-task loss for fine-tuning"""
    seq_pred = predictions['sequence_logits'].view(-1, vocab_size)
    seq_target = targets['sequences'].view(-1)
    mask_flat = mask.view(-1).bool()
    seq_loss = sequence_loss_fn(seq_pred[mask_flat], seq_target[mask_flat])
    
    coord_pred = predictions['coord_pred']
    coord_target = targets['coordinates']
    coord_mask = mask.unsqueeze(-1).expand_as(coord_pred)
    coord_loss = coord_loss_fn(coord_pred * coord_mask, coord_target * coord_mask)
    
    ss_pred = predictions['ss_pred'].view(-1, len(ss_to_idx))
    ss_target = targets['secondary_structures'].view(-1)
    ss_loss = ss_loss_fn(ss_pred[mask_flat], ss_target[mask_flat])
    
    angles_pred = predictions['angles_pred']
    angles_target = torch.stack([targets['phi_angles'], targets['psi_angles']], dim=-1)
    angles_mask = mask.unsqueeze(-1).expand_as(angles_pred)
    angles_loss = angles_loss_fn(angles_pred * angles_mask, angles_target * angles_mask)
    
    # Weighted combination - emphasize thermal stability
    total_loss = (
        1.0 * seq_loss +      # Sequence accuracy
        2.0 * coord_loss +    # 3D structure
        1.5 * ss_loss +       # Secondary structure
        1.0 * angles_loss     # Dihedral angles
    )
    
    return {
        'total_loss': total_loss,
        'sequence_loss': seq_loss,
        'coordinate_loss': coord_loss,
        'ss_loss': ss_loss,
        'angles_loss': angles_loss
    }

# Fine-tuning parameters
finetune_epochs = 10  # Fewer epochs for fine-tuning
batch_size = 2  # Smaller batch size for small dataset
num_batches = len(finetune_train_data['sequences']) // batch_size
val_num_batches = max(1, len(finetune_val_data['sequences']) // batch_size)

print(f"🔥 FINE-TUNING CONFIGURATION:")
print(f"  Epochs: {finetune_epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Training batches: {num_batches}")
print(f"  Validation batches: {val_num_batches}")
print(f"  Learning rate: 5e-5 (reduced for fine-tuning)")
print(f"  Training data: {len(finetune_train_data['sequences'])} thermophilic proteins")
print("\nStarting fine-tuning on thermophilic data...")

In [ ]:
# Fine-tuning training loop
finetune_train_losses = []
finetune_val_losses = []

for epoch in range(finetune_epochs):
    epoch_train_loss = 0.0
    num_processed = 0
    
    # Training phase
    indices = torch.randperm(len(finetune_train_data['sequences']))
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, len(finetune_train_data['sequences']))
        batch_indices = indices[start_idx:end_idx]
        
        batch_sequences = finetune_train_data['sequences'][batch_indices]
        batch_coords = finetune_train_data['coordinates'][batch_indices]
        batch_phi = finetune_train_data['phi_angles'][batch_indices]
        batch_psi = finetune_train_data['psi_angles'][batch_indices]
        batch_neighbors = finetune_train_data['neighbor_counts'][batch_indices].unsqueeze(-1)
        batch_confidence = finetune_train_data['confidence_masks'][batch_indices].unsqueeze(-1)
        batch_ss = finetune_train_data['secondary_structures'][batch_indices]
        batch_tm = finetune_train_data['tm_values'][batch_indices]
        
        batch_masks = (batch_sequences != 0).long()
        batch_angles_input = torch.stack([batch_phi, batch_psi], dim=-1)
        actual_batch_size = batch_sequences.shape[0]
        
        timesteps = sample_timesteps(actual_batch_size, T)
        noisy_sequences, noise_mask = add_noise(batch_sequences, timesteps,
                                               sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod,
                                               vocab_size)
        
        predictions = enhanced_rubisco_diffusion_forward(
            noisy_sequences, timesteps, batch_masks,
            coords=batch_coords,
            angles=batch_angles_input,
            neighbors=batch_neighbors,
            confidence=batch_confidence,
            secondary_structure=batch_ss,
            tm_values=batch_tm
        )
        
        targets = {
            'sequences': batch_sequences,
            'coordinates': batch_coords,
            'secondary_structures': batch_ss,
            'phi_angles': batch_phi,
            'psi_angles': batch_psi
        }
        
        loss_dict = compute_multi_task_loss(predictions, targets, batch_masks)
        total_loss = loss_dict['total_loss']
        
        finetune_optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(finetune_optimizer.param_groups[0]['params'], max_norm=1.0)
        finetune_optimizer.step()
        
        epoch_train_loss += total_loss.item()
        num_processed += actual_batch_size
    
    # Validation phase
    epoch_val_loss = 0.0
    val_processed = 0
    
    with torch.no_grad():
        val_indices = torch.randperm(len(finetune_val_data['sequences']))
        
        for val_batch_idx in range(val_num_batches):
            start_idx = val_batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(finetune_val_data['sequences']))
            val_batch_indices = val_indices[start_idx:end_idx]
            
            val_batch_sequences = finetune_val_data['sequences'][val_batch_indices]
            val_batch_coords = finetune_val_data['coordinates'][val_batch_indices]
            val_batch_phi = finetune_val_data['phi_angles'][val_batch_indices]
            val_batch_psi = finetune_val_data['psi_angles'][val_batch_indices]
            val_batch_neighbors = finetune_val_data['neighbor_counts'][val_batch_indices].unsqueeze(-1)
            val_batch_confidence = finetune_val_data['confidence_masks'][val_batch_indices].unsqueeze(-1)
            val_batch_ss = finetune_val_data['secondary_structures'][val_batch_indices]
            val_batch_tm = finetune_val_data['tm_values'][val_batch_indices]
            
            val_batch_masks = (val_batch_sequences != 0).long()
            val_batch_angles_input = torch.stack([val_batch_phi, val_batch_psi], dim=-1)
            val_actual_batch_size = val_batch_sequences.shape[0]
            
            val_timesteps = sample_timesteps(val_actual_batch_size, T)
            val_noisy_sequences, val_noise_mask = add_noise(val_batch_sequences, val_timesteps,
                                                           sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod,
                                                           vocab_size)
            
            val_predictions = enhanced_rubisco_diffusion_forward(
                val_noisy_sequences, val_timesteps, val_batch_masks,
                coords=val_batch_coords,
                angles=val_batch_angles_input,
                neighbors=val_batch_neighbors,
                confidence=val_batch_confidence,
                secondary_structure=val_batch_ss,
                tm_values=val_batch_tm
            )
            
            val_targets = {
                'sequences': val_batch_sequences,
                'coordinates': val_batch_coords,
                'secondary_structures': val_batch_ss,
                'phi_angles': val_batch_phi,
                'psi_angles': val_batch_psi
            }
            
            val_loss_dict = compute_multi_task_loss(val_predictions, val_targets, val_batch_masks)
            epoch_val_loss += val_loss_dict['total_loss'].item()
            val_processed += val_actual_batch_size
    
    # Calculate averages
    avg_train_loss = epoch_train_loss / num_batches
    avg_val_loss = epoch_val_loss / val_num_batches if val_num_batches > 0 else 0
    
    finetune_train_losses.append(avg_train_loss)
    finetune_val_losses.append(avg_val_loss)
    
    print(f'Fine-tune Epoch {epoch+1}/{finetune_epochs}:')
    print(f'  Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')
    print(f'  Processed: {num_processed} samples')

print("✅ Fine-tuning completed!")
print(f"Final training loss: {finetune_train_losses[-1]:.4f}")
print(f"Final validation loss: {finetune_val_losses[-1]:.4f}")

# Save fine-tuned model
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
finetuned_model_path = f"finetuned_thermophilic_rubisco_{timestamp}.pth"

torch.save({
    'embedding_state_dict': embedding.state_dict(),
    'time_mlp_state_dict': time_mlp.state_dict(),
    'transformer_encoder_state_dict': transformer_encoder.state_dict(),
    'output_layer_state_dict': output_layer.state_dict(),
    'coordinate_projection_state_dict': coordinate_projection.state_dict(),
    'angle_projection_state_dict': angle_projection.state_dict(),
    'neighbor_projection_state_dict': neighbor_projection.state_dict(),
    'confidence_projection_state_dict': confidence_projection.state_dict(),
    'ss_embedding_state_dict': ss_embedding.state_dict(),
    'tm_projection_state_dict': tm_projection.state_dict(),
    'feature_projection_state_dict': feature_projection.state_dict(),
    'coordinate_head_state_dict': coordinate_head.state_dict(),
    'secondary_structure_head_state_dict': secondary_structure_head.state_dict(),
    'angles_head_state_dict': angles_head.state_dict(),
    'pos_encoding': pos_encoding,
    'finetune_train_losses': finetune_train_losses,
    'finetune_val_losses': finetune_val_losses,
    'model_config': model_config,
    'aa_to_idx': aa_to_idx,
    'idx_to_aa': idx_to_aa,
    'ss_to_idx': ss_to_idx,
    'original_checkpoint_path': model_path,
    'finetune_data_size': len(thermo_df)
}, finetuned_model_path)

print(f"✓ Fine-tuned model saved to: {finetuned_model_path}")

# Temperature-Based Sequence Generation
**Generating RuBisCO sequences for specific target temperatures: 60°C, 70°C, 80°C, 90°C**

In [ ]:
# Temperature-conditional generation function
def generate_temperature_specific_sequences(target_tm, num_sequences=3, seq_len=None, num_steps=None):
    """
    Generate RuBisCO sequences for a specific target melting temperature
    """
    if seq_len is None:
        seq_len = max_len
    if num_steps is None:
        num_steps = T
    
    # Set model to eval mode
    embedding.eval()
    time_mlp.eval() 
    transformer_encoder.eval()
    output_layer.eval()
    coordinate_projection.eval()
    angle_projection.eval()
    neighbor_projection.eval()
    confidence_projection.eval()
    feature_projection.eval()
    coordinate_head.eval()
    secondary_structure_head.eval()
    angles_head.eval()
    
    generated_sequences = []
    
    # Use average structural features but target TM
    mean_coords = finetune_train_data['coordinates'].mean(dim=0, keepdim=True)
    mean_phi = finetune_train_data['phi_angles'].mean(dim=0, keepdim=True).unsqueeze(-1)
    mean_psi = finetune_train_data['psi_angles'].mean(dim=0, keepdim=True).unsqueeze(-1)
    mean_neighbors = finetune_train_data['neighbor_counts'].mean(dim=0, keepdim=True).unsqueeze(-1)
    mean_confidence = finetune_train_data['confidence_masks'].mean(dim=0, keepdim=True).unsqueeze(-1)
    mean_angles = torch.cat([mean_phi, mean_psi], dim=-1)
    mean_ss = torch.full((1, seq_len), 2, dtype=torch.long, device=device)  # Coil structure
    
    # TARGET TM - this is the key for conditional generation!
    target_tm_tensor = torch.full((1, seq_len), target_tm, dtype=torch.float, device=device)
    
    print(f"🌡️ Generating {num_sequences} sequences for target temperature: {target_tm}°C")
    
    for seq_idx in range(num_sequences):
        print(f"  Generating sequence {seq_idx + 1}/{num_sequences}...")
        
        # Start from random noise
        x = torch.randint(1, vocab_size, (1, seq_len), device=device)
        mask = torch.ones_like(x).long()
        
        # Reverse diffusion with target temperature conditioning
        for step, t in enumerate(reversed(range(1, num_steps + 1))):
            if step % 200 == 0:
                print(f"    Step {step}/{num_steps}, timestep {t}")
                
            t_tensor = torch.tensor([t], device=device)
            
            with torch.no_grad():
                predicted_outputs = enhanced_rubisco_diffusion_forward(
                    x, t_tensor, mask,
                    coords=mean_coords,
                    angles=mean_angles,
                    neighbors=mean_neighbors,
                    confidence=mean_confidence,
                    secondary_structure=mean_ss,
                    tm_values=target_tm_tensor  # 🔥 TARGET TEMPERATURE CONDITIONING!
                )
                
                predicted_logits = predicted_outputs['sequence_logits']
                
                if t > 1:
                    # Temperature scaling for controlled sampling
                    temperature_scale = 0.8  # Lower = more conservative
                    probs = torch.softmax(predicted_logits / temperature_scale, dim=-1)
                    x = torch.multinomial(probs.view(-1, vocab_size), 1).view(1, seq_len)
                else:
                    # Final step - use greedy decoding
                    x = torch.argmax(predicted_logits, dim=-1)
        
        generated_sequences.append(x.cpu())
        
    return generated_sequences

def decode_sequences(encoded_sequences):
    """Convert encoded sequences back to amino acid strings"""
    decoded_sequences = []
    
    for seq_tensor in encoded_sequences:
        seq_indices = seq_tensor.squeeze().tolist()
        amino_acids = []
        for idx in seq_indices:
            if idx == 0:  # PAD token
                break
            amino_acids.append(idx_to_aa[idx])
        decoded_sequences.append(''.join(amino_acids))
    
    return decoded_sequences

print("✓ Temperature-conditional generation function ready!")
print("🌡️ Will generate sequences conditioned on target melting temperatures")

In [ ]:
# Generate sequences for specific target temperatures
target_temperatures = [60, 70, 80, 90]  # °C
all_generated_data = {}

print("🌡️ GENERATING TEMPERATURE-SPECIFIC RUBISCO SEQUENCES")
print("="*60)

for target_temp in target_temperatures:
    print(f"\n🔥 TARGET TEMPERATURE: {target_temp}°C")
    print("-" * 40)
    
    # Generate 3 sequences for this temperature
    generated_encoded = generate_temperature_specific_sequences(
        target_tm=target_temp, 
        num_sequences=3, 
        num_steps=500  # Reduced steps for faster generation
    )
    
    # Decode to amino acid sequences
    generated_sequences = decode_sequences(generated_encoded)
    
    # Store results
    all_generated_data[target_temp] = {
        'sequences': generated_sequences,
        'lengths': [len(seq) for seq in generated_sequences]
    }
    
    # Display results for this temperature
    print(f"\n✅ GENERATED SEQUENCES FOR {target_temp}°C:")
    for i, seq in enumerate(generated_sequences):
        print(f"\nSequence {i+1} (Length: {len(seq)}):")
        print(f"Target Tm: {target_temp}°C")
        print(f"Sequence: {seq}")
        print("-" * 80)
    
    avg_length = np.mean(all_generated_data[target_temp]['lengths'])
    print(f"\nSummary for {target_temp}°C:")
    print(f"  Generated: {len(generated_sequences)} sequences")
    print(f"  Average length: {avg_length:.1f}")
    print(f"  Length range: {min(all_generated_data[target_temp]['lengths'])} to {max(all_generated_data[target_temp]['lengths'])}")

print("\n" + "="*60)
print("🧬 TEMPERATURE-CONDITIONAL GENERATION COMPLETE!")
print("="*60)

# Overall summary
total_sequences = sum(len(data['sequences']) for data in all_generated_data.values())
print(f"\nOverall Summary:")
print(f"  Total sequences generated: {total_sequences}")
print(f"  Target temperatures: {target_temperatures}")
print(f"  Sequences per temperature: 3")

# Display temperature comparison
print(f"\nTemperature Comparison:")
for temp in target_temperatures:
    lengths = all_generated_data[temp]['lengths']
    avg_len = np.mean(lengths)
    print(f"  {temp}°C: {len(lengths)} sequences, avg length {avg_len:.1f}")

print(f"\n🔬 These sequences are conditionally generated for specific thermal stability!")
print(f"💡 Higher target temperatures may correlate with specific amino acid patterns for thermostability.")

In [ ]:
# Save temperature-specific sequences to CSV
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f"temperature_specific_rubisco_sequences_{timestamp}.csv"

# Prepare data for CSV
csv_data = []
sequence_id = 1

for target_temp in target_temperatures:
    sequences = all_generated_data[target_temp]['sequences']
    lengths = all_generated_data[target_temp]['lengths']
    
    for i, (seq, length) in enumerate(zip(sequences, lengths)):
        csv_data.append({
            'ID': f'ThermoRuBisCO_{target_temp}C_{i+1:02d}',
            'target_temperature_C': target_temp,
            'sequence': seq,
            'length': length,
            'generation_method': f'Fine-tuned_Thermophilic_Diffusion_Model_Tm{target_temp}C',
            'sequence_number': i + 1,
            'global_id': sequence_id
        })
        sequence_id += 1

# Create DataFrame
results_df = pd.DataFrame(csv_data)

# Save to CSV
results_df.to_csv(csv_filename, index=False)

print(f"\n💾 TEMPERATURE-SPECIFIC SEQUENCES SAVED")
print(f"✓ Filename: {csv_filename}")
print(f"✓ Total records: {len(results_df)}")
print(f"✓ Columns: {list(results_df.columns)}")

# Display sample of saved data
print(f"\nSample of saved data:")
print(results_df.head())

# Summary by temperature
print(f"\nSummary by target temperature:")
temp_summary = results_df.groupby('target_temperature_C').agg({
    'sequence': 'count',
    'length': ['mean', 'min', 'max']
}).round(1)
temp_summary.columns = ['Count', 'Avg_Length', 'Min_Length', 'Max_Length']
print(temp_summary)

print(f"\n🎯 Ready for experimental validation!")
print(f"📊 Each sequence is tagged with its target melting temperature")
print(f"🧪 Researchers can test these sequences to validate temperature-specific generation")